In [1]:
import os 
import json
import math
import re
import sqlite3
import time
from collections import Counter, defaultdict
from typing import Any, TypedDict

import numpy as np
from openai import OpenAI


In [2]:
from sentence_transformers import SentenceTransformer


d:\Project\Enterprise Rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
embeding = SentenceTransformer("all-MiniLM-L6-v2")

In [4]:
client = OpenAI(base_url="https://api.groq.com/openai/v1",api_key=os.getenv("GROQ_API_KEY"))

In [5]:
CORPUS = [
    {
        "id": "pods",
        "source": "pods.md",
        "text": "A Kubernetes Pod is the smallest deployable unit. Containers in a Pod share network, storage volumes, and lifecycle. Use kubectl describe pod and kubectl logs to debug Pod behavior.",
    },
    {
        "id": "deployment",
        "source": "deployment.md",
        "text": "A Deployment manages ReplicaSets and rolling updates. Use kubectl rollout status deployment/nginx to watch progress and kubectl rollout undo deployment/nginx to roll back a bad release.",
    },
    {
        "id": "service",
        "source": "service.md",
        "text": "A Service gives stable networking for Pods. ClusterIP is internal, NodePort exposes a port on nodes, and LoadBalancer asks the cloud provider for an external endpoint.",
    },
    {
        "id": "probes",
        "source": "probes.md",
        "text": "Readiness probes decide when a Pod can receive traffic. Liveness probes restart stuck containers. Startup probes protect slow-starting applications from premature restarts.",
    },
    {
        "id": "secrets",
        "source": "secrets.md",
        "text": "Kubernetes Secrets store sensitive values such as passwords, tokens, and keys. Enable encryption at rest, restrict RBAC, and avoid printing secret data in logs.",
    },
    {
        "id": "taints",
        "source": "taints.md",
        "text": "Taints repel Pods from nodes. Tolerations allow selected Pods to schedule onto tainted nodes. A NoSchedule taint blocks Pods that lack a matching toleration.",
    },
    {
        "id": "hpa",
        "source": "hpa.md",
        "text": "The HorizontalPodAutoscaler increases or decreases replicas based on CPU, memory, or custom metrics so an application can handle more traffic automatically.",
    },
]

NOISE_DOCS = [
    {"id": "noise-cache", "source": "cache-paper.md", "text": "Cache-aware matrix multiplication improves locality in CPU memory hierarchies and reduces cache misses."},
    {"id": "noise-graph", "source": "graph-paper.md", "text": "Graph partitioning algorithms optimize edge cuts in distributed computation workloads."},
]

print(f"Inline corpus loaded: {len(CORPUS)} K8s snippets + {len(NOISE_DOCS)} noise snippets")

Inline corpus loaded: 7 K8s snippets + 2 noise snippets


In [6]:
def embed_text(texts: list[str]) -> list[list[float]]:
    response = embeding(texts)
    return [item.embedding for item in response.data]

In [7]:
def cosine(a: list[float], b: list[float]) -> float:
    av = np.array(a, dtype=float)
    bv = np.array(b, dtype=float)
    denom = np.linalg(av) * np.linalg.norm(bv)
    return float(np.dot(av, bv) / denom) if denom else 0.0

In [8]:
def dense_search(question: str, docs: list[dict[str, str]], top_k: int = 3) -> list[dict[str, Any]]:
    print(question)
    print("*"*60)
    vectors = embed_text([question] + [d['text'] for d in docs])
    print(vectors)
    print("*"*60)
    qv, doc_vector = vectors[0], vectors[1:]
    print(qv)
    print("*"*60)
    print(doc_vector)
    print("*"*60)
    ranked = []
    print(ranked)
    print("*"*60)
    for doc, dv in zip(docs, doc_vector):
        ranked.append({**doc, 'score':cosine(qv,dv)})
        print(ranked)
        print("*"*60)
    return sorted(ranked, key=lambda d: d['score'], reverse=True)[:top_k]

In [9]:
def answer_with_context(question: str, chunks:list[dict[str, Any]]) -> str:
    context = "\n\n".join(f"SOURCE: {c['source']}\n{c['text']}" for c in chunks)
    message = [
        {'role': "system", "context":"Answer only from the provided Kubernetes context. Cite source name"},
        {'role': "user", "context":f"Context: \n{context}\n\nQuestion: {question}"}
    ]
    response = client.chat.completions.create(model=os.getenv("LLM_MODEL_ANSWER"), messages=message, temperature=0)
    return response.choices[0].message.content

In [10]:
def tokenizer(text: str) -> list[str]:
    return re.findall(r"[a-zA-Z0-9_:-]+", text.lower())

In [11]:
def sparse_score(question: str, doc: dict[str, str]) -> float:
    q_terms = Counter(tokenizer(question))
    d_terms = Counter(tokenizer(doc['text']))
    return sum(q_terms[t] * d_terms[t] for t in q_terms)

In [12]:
def rrf_fuse(result_lists: list[list[dict[str, Any]]], k: int = 60, top_k: int = 3) -> list[dict[str, Any]]:
    scores: dict[str, float] = defaultdict(float)
    docs: dict[str, dict[str, Any]]  = {}
    for results in result_lists:
        for rank, doc in enumerate(results, start=1):
            scores[doc['id']] += 1 / (k + rank)
            docs[doc['id']] = doc

    fused = [{**docs[doc_id], "rrf_score": score} for doc_id, score in scores.items()]
    return sorted(fused, key=lambda d: d['rrf_score'], reverse=True)[:top_k]